In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import yt_dlp
from urllib.parse import urlparse

In [3]:
# 인스타그램 URL 확인 + shortcode 추출 함수
def is_instagram_url(url):
    parsed = urlparse(url)
    return parsed.netloc in [
        "www.instagram.com",
        "instagram.com",
        "m.instagram.com"
    ]


def get_instagram_shortcode(url):
    """
    예:
    https://www.instagram.com/p/DXWQRkxS2SK/
    -> DXWQRkxS2SK
    """
    parsed = urlparse(url)
    parts = parsed.path.strip("/").split("/")

    # 보통 구조: /p/shortcode/ 또는 /reel/shortcode/ 또는 /reels/shortcode/
    if len(parts) >= 2:
        return parts[1]

    return "unknown"

In [4]:
# 영상 다운로드 함수
def download_video(item_id, url, output_dir="videos"):
    os.makedirs(output_dir, exist_ok=True)

    if not is_instagram_url(url):
        raise ValueError("Instagram 링크만 입력할 수 있습니다.")

    shortcode = get_instagram_shortcode(url)
    base_filename = f"{item_id}_{shortcode}"

    ydl_opts = {
        "outtmpl": os.path.join(output_dir, base_filename + ".%(ext)s"),
        "format": "best[ext=mp4]/best",
        "quiet": True,
        "no_warnings": True,
        "overwrites": True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        video_path = ydl.prepare_filename(info)

    return video_path, shortcode

In [5]:
# 프레임 차이 계산 함수
def calculate_frame_diffs(video_path, resize_size=(320, 180), sample_interval=1):
    cap = cv2.VideoCapture(video_path)

    prev_gray = None
    diffs = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % sample_interval != 0:
            frame_idx += 1
            continue

        resized = cv2.resize(frame, resize_size)
        gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)

        if prev_gray is not None:
            diff = cv2.absdiff(prev_gray, gray)
            score = np.mean(diff)
            diffs.append(score)

        prev_gray = gray
        frame_idx += 1

    cap.release()
    return np.array(diffs)

In [6]:
# Adaptive threshold 계산 함수
def get_adaptive_threshold(
    video_path,
    percentile=95,
    fallback_threshold=50,
    min_threshold=15,
    max_threshold=120
):
    diffs = calculate_frame_diffs(video_path)

    if len(diffs) == 0:
        return fallback_threshold

    threshold = np.percentile(diffs, percentile)

    # 너무 낮거나 너무 높은 threshold 방지
    threshold = np.clip(threshold, min_threshold, max_threshold)

    return float(threshold)

In [16]:
# Adaptive threshold + 최소 컷 길이 적용한 이미지 추출 함수
def extract_cut_images_adaptive(
    video_path,
    item_id,
    shortcode,
    output_root="cut_images",
    percentile=95,
    min_scene_sec=0.5,
    capture_offset_sec=0,
    resize_size=(320, 180)
):
    video_folder_name = f"{item_id}_{shortcode}"
    output_dir = os.path.join(output_root, video_folder_name)
    os.makedirs(output_dir, exist_ok=True)

    # 영상별 adaptive threshold 계산
    threshold = get_adaptive_threshold(
        video_path,
        percentile=percentile
    )

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 0:
        fps = 30

    min_scene_len = int(fps * min_scene_sec)
    capture_offset_frames = int(fps * capture_offset_sec)

    ret, prev_frame = cap.read()
    if not ret:
        cap.release()
        raise ValueError("영상을 읽을 수 없습니다.")

    prev_resized = cv2.resize(prev_frame, resize_size)
    prev_gray = cv2.cvtColor(prev_resized, cv2.COLOR_BGR2GRAY)

    saved_images = []
    cut_count = 1
    frame_idx = 0
    last_cut_frame = 0

    # 첫 프레임 저장
    first_path = os.path.join(output_dir, f"{item_id}_cut_{cut_count:03d}.jpg")
    cv2.imwrite(first_path, prev_frame)
    saved_images.append(first_path)
    cut_count += 1


    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1

        resized = cv2.resize(frame, resize_size)
        gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)

        diff = cv2.absdiff(prev_gray, gray)
        diff_score = np.mean(diff)

        enough_gap = (frame_idx - last_cut_frame) >= min_scene_len

        if diff_score > threshold and enough_gap:
            # 기본값은 현재 프레임 저장
            frame_to_save = frame

            # capture_offset_sec 값이 0보다 크면,
            # 컷 감지 후 지정한 시간만큼 뒤의 프레임을 저장
            if capture_offset_frames > 0:
                target_frame_idx = frame_idx + capture_offset_frames

                # 현재 영상 위치 저장
                current_pos = cap.get(cv2.CAP_PROP_POS_FRAMES)

                # 저장하고 싶은 위치로 이동
                cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame_idx)

                ret_offset, offset_frame = cap.read()

                # offset 프레임을 정상적으로 읽었으면 그 프레임을 저장
                if ret_offset:
                    frame_to_save = offset_frame

                # 다시 원래 위치로 복귀
                cap.set(cv2.CAP_PROP_POS_FRAMES, current_pos)

            save_path = os.path.join(output_dir, f"{item_id}_cut_{cut_count:03d}.jpg")
            cv2.imwrite(save_path, frame_to_save)

            saved_images.append(save_path)
            cut_count += 1
            last_cut_frame = frame_idx

        prev_gray = gray

    cap.release()

    result = {
        "item_id": item_id,
        "shortcode": shortcode,
        "video_path": video_path,
        "output_dir": output_dir,
        "threshold": threshold,
        "fps": fps,
        "min_scene_sec": min_scene_sec,
        "num_cut_images": len(saved_images),
        "saved_images": saved_images
    }

    return result

In [70]:
# 추출할 영상에 따라 해당 코드는 영상 번호 및 링크를 변경한 후 실행시켜야합니다!
instagram_items = [
    ("0121", "https://www.instagram.com/reels/DXML1SXgZxk/"),
]

In [83]:
# 전체 실행 코드
results = []
errors = []

for item_id, url in instagram_items:
    print("=" * 60)
    print(f"[{item_id}] 처리 시작")
    print(url)

    try:
        video_path, shortcode = download_video(
            item_id=item_id,
            url=url,
            output_dir="videos"
        )

        print("영상 다운로드 완료:", video_path)

        result = extract_cut_images_adaptive(
            video_path=video_path,
            item_id=item_id,
            shortcode=shortcode,
            output_root="cut_images",
            percentile=98,
            min_scene_sec=0.2,
            capture_offset_sec=2.0
        )

        results.append({
            "item_id": item_id,
            "url": url,
            "shortcode": shortcode,
            "video_path": result["video_path"],
            "output_dir": result["output_dir"],
            "threshold": result["threshold"],
            "fps": result["fps"],
            "min_scene_sec": result["min_scene_sec"],
            "num_cut_images": result["num_cut_images"],
            "status": "success"
        })

        print(f"Adaptive threshold: {result['threshold']:.2f}")
        print(f"추출 이미지 수: {result['num_cut_images']}")
        print(f"저장 폴더: {result['output_dir']}")

    except Exception as e:
        print("오류 발생:", e)

        errors.append({
            "item_id": item_id,
            "url": url,
            "error": str(e)
        })

        results.append({
            "item_id": item_id,
            "url": url,
            "shortcode": get_instagram_shortcode(url),
            "video_path": None,
            "output_dir": None,
            "threshold": None,
            "fps": None,
            "min_scene_sec": None,
            "num_cut_images": 0,
            "status": "fail"
        })

print("=" * 60)
print("전체 처리 완료")

[0121] 처리 시작
https://www.instagram.com/reels/DXML1SXgZxk/
영상 다운로드 완료: videos\0121_DXML1SXgZxk.mp4                  
Adaptive threshold: 15.00
추출 이미지 수: 2
저장 폴더: cut_images\0121_DXML1SXgZxk
전체 처리 완료
